# 🧠 Memory Management in AI & LangChain
> **Mastering Short-Term vs. Long-Term Memory, Context Window Optimization, and Summarization Strategies in Agentic AI Applications**

---

## 📌 1. LangChain Memory Abstractions Overview

LangChain provides several built-in memory utilities designed to maintain context across conversation turns. Below is a quick overview of classic memory classes:

| Memory Class | Description |
| :--- | :--- |
| **`ConversationBufferMemory`** | Stores raw conversation history in a buffer; passes full history to the model on every call. |
| **`ConversationBufferWindowMemory`** | Keeps a sliding window of the last $K$ interactions, discarding older turns to conserve tokens. |
| **`ConversationTokenBufferMemory`** | Truncates history based on raw token count rather than fixed message counts. |
| **`ConversationSummaryMemory`** | Continuously generates and updates a running summary of the conversation history. |
| **`ConversationSummaryBufferMemory`** | Combines summary and buffer window: keeps recent messages raw while summarizing older turns. |
| **`ConversationEntityMemory`** | Extracts and tracks specific entities (people, places, concepts) throughout the chat. |
| **`VectorStoreRetrieverMemory`** | Stores memories in a vector store and performs semantic retrieval based on relevance. |
| **`CombinedMemory`** | Allows combining multiple memory implementations into a single unified memory manager. |
| **`ConversationStringBufferMemory`** | Stores history as a single formatted string buffer rather than structured message objects. |

---

## 🏗️ 2. Memory Architecture: Short-Term vs. Long-Term Memory

Modern AI Agent architectures (such as **LangGraph**) structure memory management into two main tiers:

```
Memory Management Architecture
│
├── ⚡ Short-Term Memory (Session / Thread Scope)
│      │
│      ├── Checkpointer     ──► Persists current thread state to resume chat seamlessly
│      ├── Trim             ──► Removes older messages to fit context & cut token costs
│      ├── Delete           ──► Permanently removes specific messages from state
│      ├── Summarize        ──► Compresses lengthy chat history into a concise summary
│      └── Custom Filtering ──► Applies rule-based filtering on active context
│
└── 💾 Long-Term Memory (Cross-Session / User Scope)
       │
       ├── Store.put()        ──► Saves user facts, preferences, or profile memories
       ├── Store.get()        ──► Fetches exact saved memory by key / ID
       ├── Store.search()     ──► Queries memories filtered by namespace / user_id
       ├── user_id / namespace──► Isolates memory state per user or category
       └── Semantic Retrieval ──► Uses vector embeddings to retrieve relevant memories by meaning
```

---

## 📜 3. Quick Reference: Core Operations & Functionality

### ⚡ Short-Term Memory Operations
- **`Checkpointer`**: Saves the current conversation/thread state so the ongoing chat session can be resumed seamlessly.
- **`Trim`**: Truncates older messages and forwards only recent messages to the LLM to minimize token consumption and reduce costs.
- **`Delete`**: Permanently removes targeted messages from the active memory or thread state.
- **`Summarize`**: Condenses long conversation histories into a compact, informative summary.
- **`Custom Filtering`**: Uses custom rule-based logic to specify which messages are retained in the context window.

### 💾 Long-Term Memory Operations
- **`Store.put()`**: Persists user facts, preferences, behavioral profiles, or custom memories across sessions.
- **`Store.get()`**: Fetches an exact saved memory using its unique key or ID.
- **`Store.search()`**: Searches across saved memories, supporting filtering by namespace or user scope.
- **`user_id / namespace`**: Isolates stored memories by individual user accounts, tenants, or categories.
- **`Semantic Retrieval`**: Retrieves memories based on semantic meaning and vector similarity rather than rigid keyword matching.


## 1. Fundamentals: What is Memory in AI?

### ❓ What is Memory in AI?
Memory is the foundational mechanism that enables an AI application to **store, retrieve, and reuse information** across conversation turns. It transforms a stateless Large Language Model (LLM) into an interactive, context-aware assistant capable of personalized, consistent multi-turn conversations.

---

### ⚡ The Problem: LLMs are Stateless
By default, Large Language Models are completely **stateless**. Every API invocation is treated as an isolated, independent event. If past conversation history is not explicitly included in the prompt payload, the model has zero recollection of previous exchanges.

```
Without Memory (Stateless Execution):
┌───────────┐    "My name is Sunny."     ┌───────────┐
│   User    │ ─────────────────────────► │    LLM    │ ──► "Nice to meet you!"
└───────────┘                            └───────────┘
     │                                         ▲
     │          "What is my name?"             │
     └─────────────────────────────────────────┴──► "I'm sorry, I don't know your name."

With Memory (Stateful Context Injection):
┌───────────┐    Previous Context + Query ┌───────────┐
│   User    │ ──────────────────────────► │    LLM    │ ──► "Your name is Sunny!"
└───────────┘                             └───────────┘
```

---

### 💡 Concrete Examples

#### Example 1: User Identity Retention
```
User:  "My name is Sunny."
AI:    "Nice to meet you, Sunny!"
User:  "What is my name?"
AI:    "Your name is Sunny."
```

#### Example 2: Project Context Persistence
```
User:  "I am working on an Agentic RAG system."
...
User:  "What project am I currently working on?"
AI:    "You are working on Agentic RAG!"  (Retrieved via Memory)
```

---

### 🗂️ Two Major Memory Categories

1. **Short-Term Memory**:
   - Focuses on the **current active session or conversation thread**.
   - Manages recent message history, active tool execution results, and temporary state.
   - Typically stored in-memory or in fast key-value caches (e.g., Redis, SQLite checkpointers).

2. **Long-Term Memory**:
   - Persists knowledge **across multiple sessions and long timeframes**.
   - Stores user preferences, behavioral traits, personal facts, and past project details.
   - Managed via databases, vector stores, and structured document stores.

---

### 🎯 Why Memory Management is Essential in Production

- 🔄 **Conversation Continuity**: Maintain multi-turn natural flow and handle complex follow-up questions seamlessly.
- 🎯 **Personalization**: Remember user preferences, technical stack, and personal choices across interactions.
- ⚙️ **Workflow State Tracking**: Track multi-step agentic workflows and multi-tool execution paths.
- 💰 **Token & Cost Optimization**: Prevent context overflow and control expensive token consumption.
- ⚡ **Latency Reduction**: Smaller, targeted prompt payloads lead to faster model generation times.


## 2. Environment Setup & Model Initialization

Before experimenting with memory strategies, we set up our API key environment and initialize the Google Gemini chat model (`google_genai:gemini-3.1-flash-lite`) using LangChain's unified `init_chat_model` abstraction.


In [2]:
import os
from getpass import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter GOOGLE_API_KEY: ")

print("GOOGLE_API_KEY configured:", bool(os.getenv("GOOGLE_API_KEY")))

GOOGLE_API_KEY configured: True


In [3]:
from langchain.chat_models import init_chat_model

In [4]:
model = init_chat_model("google_genai:gemini-3.1-flash-lite")

In [5]:
model.invoke([{"role": "user", "content": "Hello, how are you?"}])

AIMessage(content=[{'type': 'text', 'text': "I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EnEKbwERTTIPsw76I61guOvKtTEi1k6c38UUkIA2Q7GYDgWPzbNTBfWrFIPJ1xDirmhXuk1J0wzlcnbwqjCzr76mwJ6BrBqtrR/J5zbhe02ZEtkqkN/3Jn8EzyDGsOlkVqWKxONNEGypDbIy5CAEs2HxoQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07c37-e608-78d1-b467-8fe6ac6e9462-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 26, 'total_tokens': 33, 'input_token_details': {'cache_read': 0}})

## 3. Naive Memory: Unbounded Conversation Buffer

In this initial approach, we store **every single message** sequentially in a raw Python list (`memory = []`). On each user turn, the entire message history is passed directly into `model.invoke()`.


In [6]:
memory = []

In [7]:
def chat(user_input:str)-> str:
    memory.append(
        {
            "role": "user",
            "content": user_input
        }
    )
    
    response = model.invoke(memory)
    content = response.content if isinstance(response.content, str) else response.content[0]["text"]
    
    memory.append(
        {
            "role": "assistant",
            "content": content
        }
    )
    return content


In [8]:
def run_chat_loop():

    print("Type 'exit' or 'quit' to stop.")

    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in {"exit","quit",}:
            print("Chat ended.")
            break

        if not user_input:
            continue

        answer = chat(user_input)

        print("\nAI:", answer)

In [9]:
run_chat_loop()

Type 'exit' or 'quit' to stop.

AI: [{'type': 'text', 'text': 'Hello, Areeb! It’s a pleasure to meet you. I am doing very well, thank you for asking! How are you doing today? Is there anything I can help you with?', 'extras': {'signature': 'EnEKbwERTTIPPIHzXRuw/Cj4JQNfrVNJIxJYAG/svigYcfH09mbhJcU+KzsTZIPEqT672p9oH8EsD4jx2+6ujFBOR51ddUQXn0N66VwfsWOsPrM1kenjym/gxCrjrU2m1L8Vty+4s+gk6Rfr0zKqhpUgJg=='}}]

AI: [{'type': 'text', 'text': 'I would love to! Football (or soccer, depending on where you are!) is a massive topic with so much to talk about.\n\nTo get us started, what kind of football fan are you? Here are a few ways we could dive in:\n\n*   **The Professional Leagues:** Do you follow the Premier League, La Liga, Champions League, or maybe somewhere else?\n*   **Player Debates:** Who do you think is the G.O.A.T. (Greatest of All Time)? Or who is currently the best player in the world?\n*   **Tactics and Strategy:** Are you interested in how certain managers set up their teams, like Pep

In [10]:
memory

[{'role': 'user', 'content': 'Hello, how are you? My name is Areeb'},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'Hello, Areeb! It’s a pleasure to meet you. I am doing very well, thank you for asking! How are you doing today? Is there anything I can help you with?',
    'extras': {'signature': 'EnEKbwERTTIPPIHzXRuw/Cj4JQNfrVNJIxJYAG/svigYcfH09mbhJcU+KzsTZIPEqT672p9oH8EsD4jx2+6ujFBOR51ddUQXn0N66VwfsWOsPrM1kenjym/gxCrjrU2m1L8Vty+4s+gk6Rfr0zKqhpUgJg=='}}]},
 {'role': 'user', 'content': 'Can we discuss about football'},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'I would love to! Football (or soccer, depending on where you are!) is a massive topic with so much to talk about.\n\nTo get us started, what kind of football fan are you? Here are a few ways we could dive in:\n\n*   **The Professional Leagues:** Do you follow the Premier League, La Liga, Champions League, or maybe somewhere else?\n*   **Player Debates:** Who do you think is the G.O.A.T

In [11]:
for message in memory:
        print(message["role"], "->", message["content"])

user -> Hello, how are you? My name is Areeb
assistant -> [{'type': 'text', 'text': 'Hello, Areeb! It’s a pleasure to meet you. I am doing very well, thank you for asking! How are you doing today? Is there anything I can help you with?', 'extras': {'signature': 'EnEKbwERTTIPPIHzXRuw/Cj4JQNfrVNJIxJYAG/svigYcfH09mbhJcU+KzsTZIPEqT672p9oH8EsD4jx2+6ujFBOR51ddUQXn0N66VwfsWOsPrM1kenjym/gxCrjrU2m1L8Vty+4s+gk6Rfr0zKqhpUgJg=='}}]
user -> Can we discuss about football
assistant -> [{'type': 'text', 'text': 'I would love to! Football (or soccer, depending on where you are!) is a massive topic with so much to talk about.\n\nTo get us started, what kind of football fan are you? Here are a few ways we could dive in:\n\n*   **The Professional Leagues:** Do you follow the Premier League, La Liga, Champions League, or maybe somewhere else?\n*   **Player Debates:** Who do you think is the G.O.A.T. (Greatest of All Time)? Or who is currently the best player in the world?\n*   **Tactics and Strategy:** Are

### ⚠️ Bottlenecks of Unbounded Memory (The Full History Trap)

While passing full history is simple, it quickly breaks down as conversations grow longer:

- 📈 **Exponential Token Growth**: Every new turn sends all prior messages again, leading to quadratic token consumption.
- 💸 **Escalating Costs**: API billing scales linearly with input token count per request.
- ⏳ **Increased Latency**: Processing larger prompts adds noticeable time-to-first-token delay.
- 💥 **Context Window Limits**: Exceeding model token limits causes API errors (`ContextWindowExceededError`).

> 💡 **Architectural Solution**: We must **decouple what we store** (full audit log) from **what we send to the LLM** (active context payload).


## 4. Strategy 1: Sliding Window Memory (Buffer Window)

### 🪟 How Sliding Window Memory Works
In a **Sliding Window** strategy, we store the full conversation history in long-term storage, but only slice and supply the **most recent $N$ messages** (e.g., last 6 messages) to the LLM active context.

```
Full Stored History (Audit Log):
[ M1 | M2 | M3 | M4 | M5 | M6 | M7 | M8 | M9 | M10 ]
 \________________/   \____________________________/
  Truncated Messages      Active LLM Context Window (Last 6)
```

- 🟢 **Pros**: Extremely fast, simple to implement, keeps token usage strictly bounded.
- 🔴 **Cons**: Discards older context entirely. If the user stated their name in `M1`, the model forgets it by `M7`.


In [12]:
full_history = [
    "M1",
    "M2",
    "M3",
    "M4",
    "M5",
    "M6",
    "M7",
    "M8",
    "M9",
    "M10"
]

In [13]:
recent_context = full_history[-6:]

In [14]:
recent_context

['M5', 'M6', 'M7', 'M8', 'M9', 'M10']

In [15]:
memory = []

In [16]:
MAX_CONTEXT_MESSAGES = 6

In [17]:
def chat(user_input: str) -> str:
    memory.append(
        {
            "role": "user",
            "content": user_input
        }
    )
    recent_context = memory[-MAX_CONTEXT_MESSAGES:]
    
    response = model.invoke(recent_context)
    content = response.content if isinstance(response.content, str) else response.content[0]["text"]
    memory.append(
        {
            "role": "assistant",
            "content": content
        }
    )
    return content


In [18]:
def run_chat_loop():
    print("Type 'exit' or 'quit' to stop.")

    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in {
            "exit",
            "quit",
        }:
            print("Chat ended.")
            break

        if not user_input:
            continue

        answer = chat(user_input)

        print("\nAI:", answer)

        print("\n[Full stored messages]:", len(memory))

        print(
            "[Messages used as active context]:",
            min(
                len(memory),
                MAX_CONTEXT_MESSAGES,
            )
        )

In [19]:
run_chat_loop()

Type 'exit' or 'quit' to stop.

AI: [{'type': 'text', 'text': 'Hi Areeb! It’s nice to meet you. How are you doing today? Is there anything I can help you with?', 'extras': {'signature': 'EnEKbwERTTIPbq/qt52Ezm7COQuIiTHW7LT8y9nhtf0D9Vi+mYkE9auc6Jk4xuF9d1R/G2Ee0ZfcQaXt5SALU0oIosXGUk/ffvy7TzPlNmenX6kF3NDwrdUE7VUtk/UpF+i5ZPUeIP1eCY6mSLclUD/EBw=='}}]

[Full stored messages]: 2
[Messages used as active context]: 2

AI: [{'type': 'text', 'text': "That’s awesome, Areeb! Football (or soccer, depending on where you are!) is such a high-energy and exciting sport.\n\nTo get to know your football preferences a bit better:\n\n*   **Do you play as a hobby, or are you on a team?**\n*   **What position do you usually like to play?** (Are you a goal-scorer, a defender, or maybe the goalkeeper?)\n*   **Do you have a favorite professional player or a team that you love to watch?**\n\nI'd love to hear more about your game!", 'extras': {'signature': 'EnEKbwERTTIP4oM6ZCR+J5GYnr4SpO9uec3BF4IvCWfrzl/fM9EEzYmPs

## 5. Strategy 2: Summarization + Recent Messages (Summary Buffer)

### 🧠 How Summary Buffer Memory Works
To solve the context-loss issue of sliding windows, the **Summary Buffer** strategy maintains a hybrid memory payload:
1. **Recent Messages**: Kept verbatim (e.g., last 6 messages) for active conversation turn accuracy.
2. **Conversation Summary**: Older messages are compressed into a concise running summary by an LLM prompt.
3. **Payload Structure**: The LLM receives `SystemMessage(Current Summary)` + `Recent Messages`.

```
User Query ──► Append to Memory ──► Exceeds MAX_RECENT_MESSAGES?
                                           │
                        ┌──────────────────┴──────────────────┐
                        ▼                                     ▼
                      (Yes)                                 (No)
     Slice older messages & invoke LLM              Pass active memory
     to update conversation_summary                      directly
                        │                                     │
                        └──────────────────┬──────────────────┘
                                           ▼
                       System Message: [Current Summary]
                       + Recent Messages (Last N turns)
                                           │
                                           ▼
                                      LLM Response
```

- 🟢 **Pros**: Preserves long-term facts (user name, key decisions, project preferences) indefinitely within a fixed token budget.
- 🟢 **Cons**: Requires periodic summary generation calls to the LLM when pruning older turns.


In [32]:
memory = []

In [33]:
conversation_summary = ""

In [34]:
MAX_RECENT_MESSAGES = 6

In [35]:
full_history = []

In [36]:
def format_messages(messages):
    if not messages:
        return "No recent conversation."

    return "\n".join(
        f"{message['role']}: {message['content']}"
        for message in messages
    )

In [37]:
def create_summary(old_summary: str, old_messages: list) -> str:

    conversation_text = format_messages(old_messages)

    prompt = f"""
You maintain conversation memory for an AI assistant.

Existing summary:
{old_summary or "No previous summary."}

Older conversation messages to merge:
{conversation_text}

Create one concise updated summary.

Preserve useful information such as:
- user name
- preferences
- projects
- decisions
- important facts
- important questions and answers

Ignore greetings and unnecessary small talk.
"""

    response = model.invoke(prompt)
    content = response.content if isinstance(response.content, str) else response.content[0]["text"]

    return content.strip()


In [38]:
def compress_memory_into_summary():

    global memory
    global conversation_summary

    if len(memory) <= MAX_RECENT_MESSAGES:
        return

    old_messages = memory[:-MAX_RECENT_MESSAGES]

    memory = memory[-MAX_RECENT_MESSAGES:]

    conversation_summary = create_summary(
        conversation_summary,
        old_messages
    )

In [39]:
def chat(user_input: str) -> str:

    global memory
    global full_history

    user_message = {"role": "user","content": user_input}

    memory.append(user_message)
    full_history.append(user_message)

    messages_for_llm = [{"role": "system", "content": f"""Previous conversation summary:{conversation_summary or "No previous summary."}"""},
                        *memory
                        ]
    
    response = model.invoke(messages_for_llm)
    content = response.content if isinstance(response.content, str) else response.content[0]["text"]

    assistant_message = {"role": "assistant","content": content}

    memory.append(assistant_message)
    full_history.append(assistant_message)

    # Compress only after a complete user/assistant turn
    compress_memory_into_summary()

    return content


In [40]:
def chat_loop():

    print("Type 'exit' or 'quit' to stop.")

    while True:

        user_input = input("\nYou: ").strip()

        if user_input.lower() in {"exit","quit"}:
            print("Chat ended.")
            break

        if not user_input:
            continue

        answer = chat(user_input)

        print("\nAI:",answer)
        print("\n[Recent active messages]:",len(memory))
        print("[Full-history messages]:",len(full_history))

        if conversation_summary:

            print("\n[Current summary]:")
            print(conversation_summary)

In [41]:
chat_loop()

Type 'exit' or 'quit' to stop.

AI: Hello Areeb! It's nice to meet you. How are you doing today? Is there anything I can help you with?

[Recent active messages]: 2
[Full-history messages]: 2

AI: That’s great! Football (or soccer, depending on where you are!) is the most popular sport in the world for a reason—it's incredibly exciting.

To get to know your taste a bit better:

*   **Do you have a favorite team** that you support religiously?
*   **Is there a player you really admire** for their skills or work ethic?
*   **Do you play yourself**, or do you prefer watching matches on TV?
*   **Who is your pick for the GOAT?** (Messi vs. Ronaldo is usually the big debate!)

Let me know!

[Recent active messages]: 4
[Full-history messages]: 4

AI: That’s a classic developer preference! It’s interesting how you moved from football to programming—it sounds like you might be a fan of logic and strategy in both fields.

Choosing **Python over C++** is a very common stance, and there are some 

---

## 📊 Summary & Comparison of Memory Strategies

| Strategy | Memory Payload Sent to LLM | Token Growth | Context Retention | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Unbounded Buffer** | Full raw history | Exponential $O(N^2)$ | 100% accurate (until window overflow) | Short, multi-turn QA sessions |
| **Sliding Window** | Last $N$ recent turns | Bounded $O(1)$ | High for recent turns; zero for old turns | Simple customer support bots |
| **Summary Buffer** | Running Summary + Last $N$ turns | Bounded $O(1)$ | High for key facts & recent context | Complex agents, personal assistants |

---

### 🚀 Modern Production Recommendation (LangGraph)
In state-of-the-art Agentic AI applications built with **LangGraph**:
- Use **Checkpointers** (`MemorySaver`, `SqliteSaver`, `PostgresSaver`) for Short-Term thread persistence.
- Use **`trim_messages()`** or **Summary Nodes** to control token limits dynamically.
- Use **LangGraph Store** (`InMemoryStore`, `BaseStore`) for Long-Term cross-thread memory storage.
